# 📘 学习注释版：Gold Sales Fact

**目的：把销售业务事件与 Customer/Product Dimension 关联。**

输出：`workspace.gold.fact_sales`

重点：
- Fact 记录“发生了什么”
- 用 `customer_key` / `product_key` 关联维表
- 因为依赖两张 Dimension，所以 Fact 必须后执行


#The Transformation Logic

## 📊 学习说明：构建 Sales Fact

**Input：**
- `silver.crm_sales`
- `gold.dim_products`
- `gold.dim_customers`

**Process：**
把销售记录中的业务 Key 映射为 Dimension Key。

**Output：** `gold.fact_sales`

这也解释了为什么 Fact 必须在两张 Dimension 之后执行。


In [0]:
query = """
SELECT
    sd.order_number,
    pr.product_key,
    cu.customer_key,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.price
FROM silver.crm_sales sd
LEFT JOIN gold.dim_products pr
    ON sd.product_number = pr.product_number
LEFT JOIN gold.dim_customers cu
    ON sd.customer_id = cu.customer_id;
"""
df = spark.sql(query)


## 👀 学习说明：DataFrame Sanity Check

只显示前 10 行，快速确认当前 DataFrame：
- 字段是否正确
- 清洗是否生效
- 数据是否仍然存在

这一步不写表，只是开发时的中间检查。


In [0]:
df.limit(10).display()

#Writing Gold Table

## 💾 学习说明：把 DataFrame 持久化为 Delta Table

**Input：** 当前 `df`  
**Process：**
- `mode("overwrite")`：目标已存在时覆盖
- `format("delta")`：使用 Delta 格式
- `saveAsTable()`：注册为 Catalog Table

**Output：** `workspace.gold.fact_sales`

注意：这也是为什么 Bootcamp 可以重复运行而通常不会因为“表已存在”直接失败。


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.fact_sales")

## Sanity checks of Gold table

## ✅ 学习说明：验证 Sales Fact

重点确认：
- `customer_key`
- `product_key`
- order/sales/quantity/price

Fact 表是后续 BI / 分析的重要业务明细。


In [0]:
%sql
SELECT * FROM workspace.gold.fact_sales LIMIT 10